# ✊✌️🖐️ CNN 石頭剪刀布辨識模型
這份 Notebook 使用簡化版 CNN 模型來辨識手勢資料（0 = 石頭，2 = 剪刀，5 = 布）。資料為灰階格式的 numpy 檔案 `X.npy`, `Y.npy`，並匯出成 `.tflite` 模型以供部署在 Android 裝置上。

## 📥 載入資料與過濾 0 / 2 / 5 類別

In [ ]:
import numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# 載入 .npy 格式資料
X_all = np.load("X.npy")         # (2062, 64, 64)
Y_all = np.load("Y.npy")         # (2062, 10) - one-hot

# 將 one-hot 還原為類別索引
labels = np.argmax(Y_all, axis=1)

# 只保留類別 0, 2, 5
KEEP = [0, 2, 5]
LABEL_MAP = {0: 0, 2: 1, 5: 2}
selected = np.isin(labels, KEEP)
X_sel = X_all[selected]
y_sel = labels[selected]

# 轉換標籤為 0, 1, 2
y_mapped = np.array([LABEL_MAP[y] for y in y_sel])
y_onehot = to_categorical(y_mapped, num_classes=3)

# 灰階通道擴增 + Resize 到 96x96
X_sel = np.expand_dims(X_sel, -1)                             # (N, 64, 64, 1)
X_sel = tf.image.resize(X_sel, [96, 96]).numpy() / 255.0      # 正規化到 0~1


## 🧠 建立簡易 CNN 模型

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(96,96,1)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


## 🏋️ 模型訓練

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
X_train, X_test, y_train, y_test = train_test_split(
    X_sel, y_onehot, test_size=0.1, stratify=y_mapped
)

early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)


## 💾 匯出 TFLite 模型

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("rps_cnn.tflite", "wb") as f:
    f.write(tflite_model)

print("✅ 模型已匯出為 rps_cnn.tflite")